<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Contrôle visuomoteur

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/visual_control/pic1_rect.png", width=400px>
  <p>Vue depuis un Duckiebot lorsqu'il est centré dans sa voie.</p>
  </div>
</figure>

Nous utilisons maintenant les outils de traitement d'images que nous avons explorés pour concevoir une politique « visuo-motrice » de suivi de voie (une politique qui établit une correspondance directe entre les pixels de l'image et les valeurs de contrôle). Prenons l'image ci-dessus, prise par la caméra du Duckiebot lorsque celui-ci était centré dans sa voie. Notre objectif est de concevoir une politique de contrôle qui utilise uniquement les images transmises par la caméra du Duckiebot pour le maintenir dans sa voie lorsqu'il avance à vitesse constante.

Plutôt que de concevoir un contrôleur qui analyse l'image brute, nous allons la traiter afin d'en extraire les éléments utiles au suivi de voie. En particulier, le marquage au sol en pointillés jaunes et en traits blancs continus fournit des repères précieux pour guider le robot et le maintenir dans sa voie.

Supposons que le Duckiebot démarre au centre de la voie, face au sens de la marche. Supposons également qu'il avance à vitesse nominale. On peut imaginer utiliser la position des marquages ​​au sol (jaunes en pointillés et blancs continus) pour moduler la vitesse de rotation des roues gauche et droite et ainsi rester dans sa voie. Par exemple, si les marquages ​​jaunes en pointillés apparaissent dans les moitiés gauche et droite de l'image, on pourrait appliquer une vitesse de rotation négative (sens antihoraire) pour corriger la trajectoire.

Bien entendu, cela suppose de pouvoir détecter ces marquages ​​et d'estimer leur orientation. Il s'agit donc d'appliquer un filtre qui les fasse ressortir nettement sur l'image, tout en atténuant les autres éléments non pertinent. Les marquages ​​sont de couleur vive sur un fond sombre (la route).

On peut identifier la position des marquages ​​sur l'image en appliquant des filtres aux différences finies $h_x$ et $h_y$ (par exemple, à l'aide de l'opérateur de Sobel) pour estimer les gradients d'intensité horizontaux et verticaux comme nous l'avons vu dans [le notebook précédent](../03-Image-Filtering/image_filtering.ipynb).

Dans cet exercice, nous développerons cette idée en explorant différentes méthodes de traitement d'images afin d'extraire des informations sur la position du robot par rapport à la voie. Nous utiliserons ces résultats pour implémenter un contrôleur réactif dans l'espace image, que nous validerons sur un Duckiebot simulé et/ou réel.

In [ ]:
## Run this cell to import relevant modules
%load_ext autoreload
%autoreload 2
%matplotlib inline
%pylab inline

from matplotlib import pyplot as plt
import numpy as np
import cv2

Chargeons l'image ainsi que l'homographie de la caméra correspondante dans Python. Il s'agit de l'homographie qui associe les pixels au plan du sol dans l'image fournie ; celle dont vous avez besoin pour votre Duckiebot sera différente (vous l'avez trouvée à partir de la procédure d'étalonnage extrinsèque).

In [ ]:
# Charger l'image et générer les versions HSV et en niveaux de gris

imgbgr = cv2.imread('../../assets/images/visual_control/pic1_rect.png')

# OpenCV utilise le format BGR par défaut, tandis que matplotlib utilise le format RGB ; nous générons donc une version RGB à des fins de visualisation.
imgrgb = cv2.cvtColor(imgbgr, cv2.COLOR_BGR2RGB)

# Convertissez l'image au format HSV pour tout filtrage basé sur la couleur. Plus d'informations sur l'HSV prochainement
imghsv = cv2.cvtColor(imgbgr, cv2.COLOR_BGR2HSV)

# La plupart de nos opérations seront effectuées sur la version en niveaux de gris.
img = cv2.cvtColor(imgbgr, cv2.COLOR_BGR2GRAY)

# L'homographie image-fond associée à cette image
H = np.array([-4.137917960301845e-05, -0.00011445854191468058, -0.1595567007347241, 
              0.0008382870319844166, -4.141689222457687e-05, -0.2518201638170328, 
              -0.00023561657746150284, -0.005370140574116084, 0.9999999999999999])

H = np.reshape(H,(3, 3))
Hinv = np.linalg.inv(H)

print(Hinv)

## Trouver l'horizon

Dans Duckietown, le sol est plan. Dans d'autres domaines de la conduite autonome et de la robotique, on suppose souvent que le sol est localement plan. Nous pouvons exploiter cette propriété lors de la recherche du marquage au sol, en évitant de chercher au-dessus de l'horizon. Dans le repère du monde par rapport auquel nous avons estimé l'homographie (c'est-à-dire un repère dont l'origine est centrée entre les roues motrices, l'axe des $x$ positifs pointant vers l'avant et l'axe des $y$ positifs vers la gauche), l'horizon correspond aux grandes coordonnées $x$. Dans Duckietown, nous nous intéressons uniquement à la route située à quelques mètres devant le Duckiebot.

**Remarque :** Il n'est pas nécessaire d'inclure ceci dans l'implémentation des fonctions qui seront exécutées en simulation et sur le Duckiebot physique. Pour cette activité, le robot masquera automatiquement l'horizon.

In [ ]:
# TODO: Utilisez l'homographie pour masquer l'horizon.
# Vous pouvez y parvenir en définissant un point très éloigné sur le plan du sol,
# puis en utilisant l'inverse de l'homographie pour trouver son
# pixel associé.
# N'oubliez pas que vous devez définir le point sur le sol en coordonnées homogènes,
# puis convertir les coordonnées pour récupérer le pixel.
# Lors de la génération de masques destinés à être utilisés avec OpenCV, essayez de les
# instancier avec dtype=np.uint8

horizon = 10 # CHANGEZ-MOI. Utilisez l'homographie inverse.

print(f"horizon pixel = {horizon}")

h, w = img.shape

mask_ground = np.zeros((h, w), dtype=np.uint8)
mask_ground[int(horizon) :, :] = 1

fig = plt.figure(figsize = (30,20))
ax1 = fig.add_subplot(1,4,1)
ax1.imshow(mask_ground*img,cmap = 'gray')
ax1.set_title('Horizon Mask'), ax1.set_xticks([]), ax1.set_yticks([]);


## Détection des marquages ​​au sol à l'aide de la détection de contours de Sobel

Appliquons maintenant l'opérateur de Sobel à l'image pour identifier les gradients de l'image.

In [ ]:
# Convoluez l'image avec l'opérateur de Sobel (filtre) pour calculer les dérivées numériques selon les directions x et y.
sobelx = cv2.Sobel(img,cv2.CV_64F,1,0)
sobely = cv2.Sobel(img,cv2.CV_64F,0,1)

# Calculer l'amplitude des gradients
Gmag = np.sqrt(sobelx*sobelx + sobely*sobely)

# Calculer l'orientation des gradients
Gdir = cv2.phase(np.array(sobelx, np.float32), np.array(sobely, dtype=np.float32), angleInDegrees=True)

fig = plt.figure(figsize = (30,20))
ax1 = fig.add_subplot(1,4,1)
ax1.imshow(img,cmap = 'gray')
ax1.set_title('Original'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,4,2)
ax2.imshow(sobelx,cmap = 'gray')
ax2.set_title('Sobel X'), ax2.set_xticks([]), ax2.set_yticks([])
ax3 = fig.add_subplot(1,4,3)
ax3.imshow(sobely,cmap = 'gray')
ax3.set_title('Sobel Y'), ax3.set_xticks([]), ax3.set_yticks([]);
ax4 = fig.add_subplot(1,4,4)
ax4.imshow(np.uint8(Gmag))
ax4.set_title('Gradient Magnitude'), ax4.set_xticks([]), ax4.set_yticks([]);

Le filtrage de l'image a accentué le marquage au sol, mais a également mis en évidence d'autres zones de variations d'intensité rapides. Il s'agit notamment des canards, des bâtiments de Duckietown, de la chaussée et d'éléments d'arrière-plan (par exemple, les lumières dans le coin supérieur droit de l'image). Notez que nous utilisons une palette de couleurs non en niveaux de gris pour les amplitudes de gradient afin de mieux faire ressortir les zones accentuées par l'opérateur de Sobel.

Nous pouvons atténuer l'apparition de certains de ces éléments en floutant préalablement le filtre à l'aide d'un noyau gaussien.

## Brouiller une image avec un Gaussien (*Gaussian blurring*) 

L'opérateur de Sobel détecte le bruit et la texture de l'image, même s'ils ne nous intéressent pas. On peut flouter l'image avec un noyau gaussien pour réduire ces détections non pertinent.

En vous appuyant sur votre expérience de l'exercice précédent, identifiez un paramètre pour le variance du noyau gaussien qui supprime le bruit et la texture locale (par exemple, celle de la surface de la route), sans sacrifier trop de contenu valide (à savoir, les bords associés au marquage des voies).

In [ ]:
# TODO: Identifiez un paramètre pour l'écart type qui élimine le bruit sans pour autant supprimer trop de contenu valide.
sigma = 1 # CHANGEZ-MOI

# Lisser l'image à l'aide d'un noyau gaussien
img_gaussian_filter = cv2.GaussianBlur(img,(0,0), sigma)

# Visualisez l'image filtrée à côté de l'image originale.
fig = plt.figure(figsize = (20,20))
ax1 = fig.add_subplot(1,2,1)
ax1.imshow(img,cmap = 'gray')
ax1.set_title('Original'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(img_gaussian_filter,cmap = 'gray')
ax2.set_title('Gaussian Filter (Sigma = ' + str(sigma) +')'), ax2.set_xticks([]), ax2.set_yticks([]);

Examinons maintenant les gradients de l'image floue. Pour cet exercice, testez différentes valeurs de l'écart type et observez leur influence sur les gradients. Comparez notamment les gradients pour deux valeurs extrêmes de l'écart type (par exemple, σ = 1 et σ = 10).

In [ ]:
# Convoluer l'image avec l'opérateur de Sobel (filtre) pour calculer les dérivées numériques selon les directions x et y

sobelx = cv2.Sobel(img_gaussian_filter,cv2.CV_64F,1,0)
sobely = cv2.Sobel(img_gaussian_filter,cv2.CV_64F,0,1)

# Calculer l'amplitude des gradients
Gmag = np.sqrt(sobelx*sobelx + sobely*sobely)

# Calculer l'orientation des gradients
Gdir = cv2.phase(np.array(sobelx, np.float32), np.array(sobely, dtype=np.float32), angleInDegrees=True)

fig = plt.figure(figsize = (30,20))
ax1 = fig.add_subplot(1,4,1)
ax1.imshow(img_gaussian_filter,cmap = 'gray')
ax1.set_title('Gaussian Blur'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,4,2)
ax2.imshow(sobelx,cmap = 'gray')
ax2.set_title('Sobel X'), ax2.set_xticks([]), ax2.set_yticks([])
ax3 = fig.add_subplot(1,4,3)
ax3.imshow(sobely,cmap = 'gray')
ax3.set_title('Sobel Y'), ax3.set_xticks([]), ax3.set_yticks([]);
ax4 = fig.add_subplot(1,4,4)
ax4.imshow(np.uint8(Gmag))
ax4.set_title('Gradient Magnitude'), ax4.set_xticks([]), ax4.set_yticks([]);

## Éliminer les contours faibles

En choisissant judicieusement l'écart type, nous avons atténué le bruit et la texture de l'image (par exemple, la route), réduisant ainsi le nombre de contours candidats tout en préservant la plupart de ceux correspondant au marquage au sol.

En supposant que les contours associés au marquage au sol sont plus marqués que la plupart des autres contours de l'image (en termes d'amplitude des gradients), nous pouvons éliminer davantage les éléments non pertinent en ne conservant que les contours dont l'amplitude du gradient est supérieure à un seuil.

Nous examinerons l'histogramme des amplitudes de gradient afin de mieux comprendre leur distribution. À partir de cet histogramme, nous pourrons choisir un seuil d'amplitude qui nous permettra de filtrer les contours les plus faibles.

In [ ]:
# Visualiser l'histogramme en fonction des amplitudes du gradient
fig = plt.figure(figsize = (5,5))
ax1 = fig.add_subplot(1,1,1)
ax1.hist((Gmag).flatten(), bins=50)
ax1.set_title('Histogramme de magnitude du gradient');

In [ ]:
# TODO: Utilisez l'histogramme ci-dessus pour choisir le seuil minimal de magnitude du gradient.
# Les arêtes dont la magnitude du gradient est inférieure à ce seuil seront filtrées.


threshold = np.random.rand(1,1) # CHANGEZ-MOI



mask_mag = (Gmag > threshold)

fig = plt.figure(figsize = (20,10))
ax1 = fig.add_subplot(1,2,1)
ax1.imshow(Gmag, cmap = 'gray')
ax1.set_title('Gradient Magnitude'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(mask_mag*Gmag,cmap = 'gray')
ax2.set_title('Gradient Magnitude (Thresholded)'), ax2.set_xticks([]), ax2.set_yticks([]);

## Masquage basé sur la couleur

Après avoir identifié un ensemble de contours candidats, nous pouvons concevoir des masques isolant les contours associés aux marquages ​​au sol jaunes en pointillés et blancs continus.

La méthode la plus simple consiste à créer des masques filtrant les pixels dont la couleur diffère de celle des lignes jaunes et blanches.

Pour cette procédure, il est d'usage de convertir l'image RGB en un espace colorimétrique alternatif : l'espace colorimétrique << Hue-Saturation-Value >> (HSV).

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/visual_control/RGB_HSV.png", width=400px>
  <p>Les espaces colorimétriques RGB et HSV.</p>
  </div>
</figure>

La raison est simple : dans l’espace colorimétrique HSV, toutes les informations de couleur sont contenues dans un seul canal, la teinte (H). Ainsi, dans de nombreux cas, le filtrage des couleurs peut se faire uniquement à l’aide de ce canal. Le blanc fait exception, comme nous le verrons.

Il est possible de définir des limites supérieures et inférieures pour les valeurs HSV des deux lignes. Une solution consiste à utiliser [ce sélecteur de couleurs en ligne](https://pinetools.com/image-color-picker), qui permet d’obtenir les valeurs HSV de pixels spécifiques. Capturez une image ou importez-la dans cet outil, puis utilisez-le pour interroger les valeurs HSV des différents pixels.

**Remarque concernant les conventions de limit HSV :** Lorsque vous utilisez ces limites, n'oubliez pas qu'OpenCV utilise la convention $\textrm{H} \in [0, 179]$, $\textrm{S} \in [0, 255]$ et $\textrm{V} \in [0, 255]$, tandis que le sélecteur de couleurs en ligne mentionné ci-dessus utilise la convention $\textrm{H} \in [0, 255]$, $\textrm{S} \in [0, 100]$ et $\textrm{V} \in [0, 100]$. Vous devrez adapter les valeurs en conséquence.

**Remarque concernant l'éclairage** : Bien que l'espace colorimétrique HSV offre une meilleure représentation (par exemple, comparé au RGB) pour la détection basée sur la couleur, l'apparence des marquages ​​jaunes et blancs des voies peut varier en fonction de l'éclairage, des ombres, etc. Afin d'améliorer la généralisation de ces limites, il est recommandé de prendre également en compte l'image `../../assets/images/visual_control/pic3_rect.png` et de définir des limites adaptées aux deux.


In [ ]:
# À l'aide de l'outil ci-dessus, nous pouvons identifier les limites comme suit :
# TODO : Identifier les limites HSV inférieures et supérieures pour les marquages ​​blancs et jaunes des voies
# Ces valeurs représentent la plage maximale ; elles ne filtrent donc rien

white_lower_hsv = np.array([0, 0, 0])         # CHANGE ME
white_upper_hsv = np.array([179, 255, 255])   # CHANGE ME
yellow_lower_hsv = np.array([0, 0, 0])        # CHANGE ME
yellow_upper_hsv = np.array([179, 255, 255])  # CHANGE ME

mask_white = cv2.inRange(imghsv, white_lower_hsv, white_upper_hsv)
mask_yellow = cv2.inRange(imghsv, yellow_lower_hsv, yellow_upper_hsv)

fig = plt.figure(figsize = (20,10))
ax1 = fig.add_subplot(1,3,1)
ax1.imshow(imgrgb)
ax1.set_title('Original'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,3,2)
ax2.imshow(imgrgb)
ax2.imshow(mask_white, cmap='jet', alpha=0.5)
ax2.set_title('Filtered by Color (White)'), ax2.set_xticks([]), ax2.set_yticks([]);
ax3 = fig.add_subplot(1,3,3)
ax3.imshow(imgrgb)
ax3.imshow(mask_yellow, cmap='jet', alpha=0.5)
ax3.set_title('Filtered by Color (Yellow)'), ax3.set_xticks([]), ax3.set_yticks([]);

## Masquage d'orientation des contours

Nous allons maintenant explorer différentes méthodes pour masquer les bords indésirables en fonction de leur orientation dans l'image.

En supposant que le Duckiebot n'ait pas trop dévié de sa voie, la majeure partie de la ligne jaune pointillée devrait se situer dans la moitié gauche de l'image, tandis que la majeure partie de la ligne blanche continue devrait se situer dans la moitié droite. Nous pouvons ainsi analyser séparément les marquages ​​de voie jaune pointillé et blanc continu.

**Remarque** : Bien que ce masquage puisse sembler pertinent lorsque le Duckiebot est orienté dans le sens de la marche, il peut s'avérer inadapté si son orientation est incorrecte. Dans ce cas, on peut s'attendre à ce que certains marquages ​​de voie jaune pointillé se trouvent dans la moitié droite de l'image, ou inversement. Ces observations constituent des indices précieux. Ainsi, lorsque vous expérimentez avec les images incluses, ainsi qu'en simulation et sur votre Duckiebot physique, vous pouvez essayer de désactiver ces masques.


In [ ]:
# Créons des masques pour les moitiés gauche et droite de l'image.
width = img.shape[1]
mask_left = np.ones(sobelx.shape)
mask_left[:,int(np.floor(width/2)):width + 1] = 0
mask_right = np.ones(sobelx.shape)
mask_right[:,0:int(np.floor(width/2))] = 0

Le contour intérieur de la ligne jaune pointillée correspond à un gradient négatif selon les axes $x$ et $y$, tandis que le bord intérieur de la ligne blanche continue correspond à un gradient positif selon l'axe $x$ et négatif selon l'axe $y$. On peut s'en servir pour masquer plus finement les contours indésirables.

In [ ]:
# Dans la moitié gauche de l'image, nous nous intéressons à la moitié droite de la ligne jaune pointillée, qui correspond à des dérivées négatives selon x et y.
# Dans la moitié droite de l'image, nous nous intéressons à la moitié gauche de la ligne blanche continue, qui correspond à une dérivée positive selon x et une dérivée négative selon y.
# Générez un masque qui identifie les pixels en fonction du signe de leur dérivée selon x.

mask_sobelx_pos = (sobelx > 0)
mask_sobelx_neg = (sobelx < 0)
mask_sobely_pos = (sobely > 0)
mask_sobely_neg = (sobely < 0)

## Combiner les masques

Maintenant, combinons ces masques pour observer leur effet sur la détection des contours, mais examinons d'abord les effets des masques basés sur le gradient avant d'inclure ceux basés sur la couleur.

In [ ]:
# Combinons ces masques avec le masque de magnitude du gradient

mask_left_edge = mask_ground * mask_left * mask_mag * mask_sobelx_neg * mask_sobely_neg
mask_right_edge = mask_ground * mask_right * mask_mag * mask_sobelx_pos * mask_sobely_neg

fig = plt.figure(figsize = (30,10))
ax1 = fig.add_subplot(2,4,1)
ax1.imshow(img_gaussian_filter,cmap = 'gray')
ax1.set_title('Image brouiller'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(2,4,2)
ax2.imshow(mask_mag*Gmag,cmap = 'gray')
ax2.set_title('Magnitude du gradient (masquée)'), ax2.set_xticks([]), ax2.set_yticks([])
ax3 = fig.add_subplot(2,4,3)
ax3.imshow(Gmag * mask_left_edge, cmap = 'gray')
ax3.set_title('Bord de voie (gauche)'), ax3.set_xticks([]), ax3.set_yticks([])
ax4 = fig.add_subplot(2,4,4)
ax4.imshow(Gmag * mask_right_edge,cmap = 'gray')
ax4.set_title('Bord de voie (droit)'), ax4.set_xticks([]), ax4.set_yticks([]);

ax6 = fig.add_subplot(2,4,6)
ax6.imshow(mask_mag*Gmag,cmap = 'gray')
ax6.set_title('Magnitude du gradient (masquée)'), ax6.set_xticks([]), ax6.set_yticks([])
ax7 = fig.add_subplot(2,4,7)
ax7.imshow(img_gaussian_filter,cmap = 'gray')
ax7.imshow(Gmag * mask_left_edge, cmap='jet', alpha=0.5)
ax7.set_title('Bord de voie (gauche)'), ax7.set_xticks([]), ax7.set_yticks([])
ax8 = fig.add_subplot(2,4,8)
ax8.imshow(img_gaussian_filter,cmap = 'gray')
ax8.imshow(Gmag * mask_right_edge, cmap='jet', alpha=0.5)
ax8.set_title('Bord de voie (droit)'), ax8.set_xticks([]), ax8.set_yticks([]);

On constate que l'utilisation conjointe du seuillage et du masquage basé sur les gradients de l'image a permis d'éliminer de nombreux contours indésirable. Cependant, les gradients « Bord de voie (gauche) » incluent des contours qui ne correspondent pas au marquage au sol (par exemple, la transition claire-foncée en bord de route). Les gradients « Bord de voie (droite) » isolent mieux le contour formé par la ligne blanche continue.

### Orientations dominantes du gradient

Nous souhaitons maintenant estimer l'orientation des gradients pour les marquages ​​au sol jaunes en pointillés et blancs en trait plein. Pour ce faire, examinons l'histogramme des gradients des images des bords gauche et droit.

In [ ]:
# Appliquons maintenant le masque à nos directions de dégradé.
fig = plt.figure(figsize = (10,5))
ax1 = fig.add_subplot(1,2,1)
ax1.hist(np.extract(mask_left_edge, Gdir).flatten(), bins=30)
ax1.set_title('Histogramme de direction du gradient (bord gauche)')
ax2 = fig.add_subplot(1,2,2)
ax2.hist(np.extract(mask_right_edge, Gdir).flatten(), bins=30)
ax2.set_title('Histogramme de direction du gradient (bord droit)');

En observant l'histogramme des gradients du « contour droit », on constate que la grande majorité des bords présentent une orientation d'environ 315 degrés, tandis que quelques rares exceptions affichent d'autres orientations. Ceci est cohérent avec les gradients visualisés précédemment dans les images « Bord de voie (droite) ». L'orientation à environ 315 degrés correspond au bord dominant associé au marquage blanc continu de la voie (c'est-à-dire que le gradient pointe vers le haut à droite).

Intégrons maintenant les masques de couleur.

In [ ]:
# Générons l'ensemble complet des masques, y compris ceux basés sur la couleur.
mask_left_edge = mask_ground * mask_left * mask_mag * mask_sobelx_neg * mask_sobely_neg * mask_yellow
mask_right_edge = mask_ground * mask_right * mask_mag * mask_sobelx_pos * mask_sobely_neg * mask_white

fig = plt.figure(figsize = (30,10))
ax1 = fig.add_subplot(2,4,1)
ax1.imshow(img_gaussian_filter,cmap = 'gray')
ax1.set_title('Image brouiller'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(2,4,2)
ax2.imshow(mask_mag*Gmag,cmap = 'gray')
ax2.set_title('Magnitude du gradient (masquée)'), ax2.set_xticks([]), ax2.set_yticks([])
ax3 = fig.add_subplot(2,4,3)
ax3.imshow(Gmag * mask_left_edge, cmap = 'gray')
ax3.set_title('Bord de voie (gauche)'), ax3.set_xticks([]), ax3.set_yticks([])
ax4 = fig.add_subplot(2,4,4)
ax4.imshow(Gmag * mask_right_edge,cmap = 'gray')
ax4.set_title('Bord de voie (droit)'), ax4.set_xticks([]), ax4.set_yticks([]);

ax6 = fig.add_subplot(2,4,6)
ax6.imshow(mask_mag*Gmag,cmap = 'gray')
ax6.set_title('Magnitude du gradient (masquée)'), ax6.set_xticks([]), ax6.set_yticks([])
ax7 = fig.add_subplot(2,4,7)
ax7.imshow(img_gaussian_filter,cmap = 'gray')
ax7.imshow(Gmag * mask_left_edge, cmap='jet', alpha=0.5)
ax7.set_title('Bord de voie (gauche)'), ax7.set_xticks([]), ax7.set_yticks([])
ax8 = fig.add_subplot(2,4,8)
ax8.imshow(img_gaussian_filter,cmap = 'gray')
ax8.imshow(Gmag * mask_right_edge, cmap='jet', alpha=0.5)
ax8.set_title('Bord de voie (droit)'), ax8.set_xticks([]), ax8.set_yticks([]);

On constate que l'utilisation de masques de couleur a permis d'éliminer la plupart, voire la totalité, des valeurs aberrantes, mais aussi certaines limites valides, notamment celles associées aux marquages ​​de voie blancs.

Examinons les histogrammes correspondants.

In [ ]:
# Appliquons maintenant l'ensemble des masques à nos directions de dégradé.
fig = plt.figure(figsize = (10,5))
ax1 = fig.add_subplot(1,2,1)
ax1.hist(np.extract(mask_left_edge, Gdir).flatten(), bins=30)
ax1.set_title('Histogramme de direction du gradient (bord gauche)')
ax2 = fig.add_subplot(1,2,2)
ax2.hist(np.extract(mask_right_edge, Gdir).flatten(), bins=30)
ax2.set_title('Histogramme de direction du gradient (bord droit)');

Conformément à ce qui précède, l'intégration des masques de couleur a permis d'éliminer efficacement le grand nombre de contours qui généraient un mode dominant autour de 260 degrés dans l'histogramme précédent pour l'arête gauche. Désormais, chaque contour possède un seul mode dominant.

**Remarque** : Vous vous demandez peut-être pourquoi nous n'utilisons pas ces orientations de gradient pour le contrôle. En effet, on pourrait s'attendre à ce que les variations d'orientation des gradients associés aux marquages ​​de voie gauche et droite permettent de ramener le véhicule dans sa voie. Cependant, les orientations dans l'espace image ne varient pas significativement lorsque l'orientation du véhicule par rapport à la voie change, ce qui implique que le signal utile est ici moins important qu'on pourrait le penser.

# 💻 🚙 Écrivez la fonction de suivi de voie

Maintenant que nous comprenons comment détecter les marquages de voie gauche (jaune pointillé) et droite (blanc continu), nous pouvons intégrer ces composantes dans un contrôleur réactif visant à maintenir le Duckiebot dans sa voie.

En particulier, si l’on commande au Duckiebot d’avancer à une vitesse fixe, on peut imaginer contrôler la direction (vitesse angulaire) comme suit :


```python
steering  = np.sum( STEER_LEFT_LM * left_lane_markings_img) + np.sum( STEER_RIGHT_LM * right_lane_markings_img)
```

où `STEER_LEFT_LM` et `STEER_RIGHT_LM` sont des matrices de pondération qui associent les positions dans l’espace image des marquages de voie gauche (jaune pointillé) détectés aux commandes de direction (vitesse angulaire).

Dans ce contexte, nous vous demandons de définir deux ensembles de fonctions :

1. Un ensemble de deux fonctions qui définissent ces matrices de pondération ;

2. Une fonction qui prend en entrée l’image provenant de la caméra du Duckiebot et produit en sortie deux images, l’une correspondant aux détections dans l’espace image des marquages de voie gauche (jaune pointillé), et l’autre aux marquages de voie droite (blanc continu).

Ces fonctions seront ensuite combinées pour contrôler le Duckiebot.

## Définir les matrices de marquage des voies de gauche et de droite

Nous contrôlerons la direction en fonction d'une combinaison pondérée des marquages ​​de voie gauche et droite. Implémentez les fonctions `get_steer_matrix_left_lane_markings()` et `get_steer_matrix_right_lane_markings()` dans le fichier [visual_servoing_solution.py](../../packages/visual_lane_servoing/include/visual_servoing_solution.py). Ces fonctions créent des matrices de pondération pour les marquages ​​de voie gauche et droite, respectivement. Ces matrices seront multipliées par les détections et détermineront la réaction du robot (virage à gauche ou à droite). Intuitivement, les marquages ​​de voie gauche (jaunes) « poussent » le robot à tourner à droite, et les marquages ​​de voie droite (blancs) à tourner à gauche. Il est important de réfléchir à l'intensité de la « poussée » appliquée par chaque détection. Les détections les plus à gauche doivent-elles exercer une force plus ou moins importante sur le robot ?

Vous pouvez utiliser les outils suivants pour visualiser vos matrices de direction.

In [ ]:
# TODO - définir la matrice de braquage gauche qui sera utilisée pour :

# pondérer les détections de la voie jaune

width = img.shape[1] // 2
height = img.shape[0]

steer_matrix_left_lane = np.zeros((height, img.shape[1]))

# Créer une rampe ascendante: 0 → width-1
steer_unit = np.arange(width, dtype=float)

# Normaliser (éviter la division par zéro)

max_val = steer_unit.max()
if max_val != 0:
    steer_unit /= max_val


steer_matrix_left_lane[:, :width] = 1 # CHANGEZ-MOI


imshow(steer_matrix_left_lane)

In [ ]:
# TODO : définir la matrice de direction appropriée qui sera utilisée pour
# pondérer les détections de la voie jaune

width = img.shape[1] // 2
height = img.shape[0]

# Préparer la sortie
steer_matrix_right_lane = np.zeros((height, img.shape[1]))

# Créer une rampe ascendante de width → 1
steer_unit = np.arange(width, 0, -1).astype(float)

# Normaliser (éviter la division par zéro)
max_val = steer_unit.max()
if max_val != 0:
    steer_unit /= max_val

steer_matrix_right_lane[:, width:] = 1 # CHANGEZ-MOI


imshow(steer_matrix_right_lane)


## Détection des marquages ​​au sol des voies de gauche et de droite

En vous basant sur les informations précédentes, implémentez la fonction `detect_lane_markings()` dans le fichier [visual_servoing_solution.py](../../packages/visual_lane_servoing/include/visual_servoing_solution.py). Cette fonction prend en entrée l'image provenant de la caméra du Duckiebot (dans l'espace colorimétrique BGR) et produit deux images : l'une correspondant au marquage de la voie de gauche (en pointillés jaunes) et l'autre à celui de la voie de droite (en trait plein blanc). Ces images peuvent être des masques binaires, comme ceux développés précédemment.

Ces deux images de marquage seront ensuite utilisées, conjointement avec les matrices de pondération définies précédemment, pour piloter le Duckiebot.

### Testez la fonction `detect_lane_markings()` et le contrôleur

Comme nous l'avons vu, les tests unitaires sont précieux pour confirmer qu'un morceau de code fonctionne comme prévu avec les entrées attendues.

Voyons si la fonction que vous avez écrite ci-dessus réussit le test suivant !

Deux images différentes, avec des conditions d'éclairage légèrement différentes, seront présentées. Votre politique devrait prédire un angle de braquage assez faible pour ces images (ne vous inquiétez pas trop si ce n'est pas parfait… il y a des éléments perturbateurs assez gênants, comme des canards jaunes et blancs !). Dans le troisième cas, le Duckiebot est tourné vers la droite ; votre politique devrait alors prédire un angle de braquage positif (virage vers la gauche). Les valeurs renvoyées sont le produit scalaire brut de votre masque avec votre matrice de braquage (les nombres seront donc élevés). Lors de l'exécution de la politique, nous normaliserons ces valeurs, puis les convertirons en angles de braquage. L'important à ce stade est de s'assurer que la valeur obtenue dans le troisième cas est nettement supérieure à celles des deux premiers.


**Remarque : il se peut que vous deviez recharger le noyau pour que l’ordinateur portable prenne en compte les modifications apportées à l’aide du bouton « Restart » en haut.**

In [ ]:
from unit_tests import UnitTestDLM
import visual_lane_servoing.include.visual_servoing_solution as solution

# Le test fournit une image à votre fonction detect_lane_markings
# et visualise les masques gauche et droit qu'elle a produits.

UnitTestDLM(solution.detect_lane_markings, solution.get_steer_matrix_right_lane_markings, solution.get_steer_matrix_left_lane_markings)

## Testez le contrôler visuo-moteur sur le Duckiebot 

Vous pouvez suivre la procédure décrite dans le [README](../../README.md) pour tester votre code sur votre Duckiebot (soit réel ou virtuel - Il sera probablement plus facile et plus rapide de commencer par tester dans la Duckiematrix.)

Dans Duckiematrix, vous devriez pouvoir faire effectuer le tour de boucle à votre Duckiebot de manière assez fluide. Ce sera peut-être un peu plus difficile avec le vrai Duckiebot, mais vous devriez au moins pouvoir démontrer qu'il essayait de tourner dans la bonne direction.



## 💡 En réfléchissant à l'expérience

La stratégie décrite ci-dessus constitue une première approche raisonnable pour guider votre Duckiebot et le maintenir dans sa voie. Selon votre implémentation et votre environnement, vous avez peut-être même constaté qu'il parvient assez bien à rester dans sa voie, même dans les virages. Cependant, comme vous l'avez probablement remarqué, cette approche présente plusieurs limitations :

* Notre stratégie de détection des marquages ​​au sol et de conception des matrices de pondération limite les géométries routières que le Duckiebot peut gérer. S'il peut se débrouiller correctement dans les courbes douces, il aura probablement du mal à gérer les virages serrés ou les intersections en T, entre autres ;

* L'algorithme de détection est sensible aux variations d'éclairage, ce qui peut entraîner des détections erronées (par exemple, des pixels valides sont filtrés par un masque de couleur), et aux interférences, ce qui peut provoquer des détections erronées (par exemple, votre Duckiebot peut interpréter incorrectement des marquages ​​au sol sur le mur, sur les canards ou ailleurs dans la ville), ce qui le fait se diriger de manière imprévisible.

Nous espérons néanmoins que cette présentation vous a permis de découvrir certains concepts fondamentaux du filtrage d'images et leur application au contrôle des mouvements de votre Duckiebot. Nous espérons également qu'elle vous a procuré une expérience concrète des difficultés liées à l'extraction d'informations à partir d'images et à leur utilisation pour le contrôle, ce qui devrait préparer le terrain pour les prochains sujets abordés dans ce cours.